# PersonaPlex + IMTalker Live Avatar Server — RunPod RTX 5090 Deployment (Updated Code)

This notebook deploys and launches the **updated live PersonaPlex + IMTalker
`try_vad2` 8998 server**, as documented in `live_8998.md` (the authoritative
deployment document for this exact package). It is the counterpart of the old
`RunPod_RTX5090_PersonaPlex_IMTalker_Live_fixed.ipynb` notebook, rewritten
against the **new** `speech2avatar-new-code` package layout:

```
PersonaPlex response audio + hidden states
-> unitalk last-layer adapter (lookahead window)
-> IMTalker fullgen static-head generator (2s chunks)
-> IMTalker renderer (FP32)
-> cached eye-blink motion-map composite
-> browser websocket video/audio (Opus)
```

Unlike the old deployment, this package ships **no vendored `personaplex/`
source folder** and **no top-level `scripts/download_live_assets.sh`** —
instead it ships two self-contained shell scripts that this notebook drives
rather than reimplements:

- `prepare_imtalker_personaplex.sh` — installs system packages, creates the
  `.venv`, installs the pinned `torch==2.8.0+cu128` stack and all pinned
  Python dependencies, authenticates to Hugging Face, downloads every
  checkpoint/asset (IMTalker renderer + wav2vec2, 2s generator + adapter,
  blink motion, PersonaPlex bnb4 weights + Mimi/tokenizer + voices), installs
  PersonaPlex's bundled Moshi package (`--no-deps`), and runs the deployment
  preflight (`run_imtalker_personaplex.sh --check-only`, including per-file
  SHA-256 verification of the "protected" runtime files).
- `run_imtalker_personaplex.sh` — re-runs the same preflight, then launches
  the live server (`IMTalker/imtalker_personaplex_try_vad2_8998.py`) in the
  foreground on port **8998**.

This build additionally carries the online-search stack ported from the old
pipeline, plus structured logging and the latency fixes:

- **Online search with routing.** Every question is routed first. If it needs no
  live data the model answers from its own knowledge and *nothing* is injected,
  so that path stays as fast as the plain conversational server. If it does, a
  thinking sound plays while the search runs, the results are summarized into a
  single clean spoken sentence, and that sentence is injected as a `<ref>` block
  into the live context (the conversation is never reset).
- **Reference LoRA.** `Darknsu/helium_lora_v1`, applied unmerged (QLoRA-style)
  onto the 4-bit base *before* CUDA-graph capture. This is what makes context
  injection actually work -- the base model does not reliably act on injected
  text on its own.
- **Spoken-form summaries.** Injected text is normalized to what a person would
  say: no symbols or markup, and numbers written in words
  (`$325` -> "Three hundred twenty-five dollars", `€325` -> "Three hundred
  twenty-five euros").
- **Two log files, millisecond timestamps.** `system_<session>.log` (models,
  sources, LoRA paths, video config, GPU, startup timing) and
  `detailed_<session>.log` (per turn: the question, the search decision and why,
  the query, the results, the summary, exactly what was injected, the reply, and
  the duration of every stage).
- **Latency.** A microphone-backlog cap, cheaper render batching, and an opt-in
  incremental publish mode. See the Parameters cell.

This notebook performs, in order:

1. Validates the `speech2avatar-new-code` project layout under `PROJECT_ROOT`
   (uploading/unzipping it if not already present).
2. Runs `prepare_imtalker_personaplex.sh --hf-token <token>` (token entered
   securely, never hardcoded) — this single script handles apt packages, the
   venv, pinned PyTorch, pinned Python deps, all Hugging Face downloads, the
   Moshi install, and the built-in preflight/checksum verification.
3. Confirms the GPU is an RTX 5090 (or CUDA-capable equivalent) both before
   and after installation.
4. Launches `run_imtalker_personaplex.sh` in the background, waits for the
   Uvicorn "listening" banner and a successful HTTP health check, and prints
   the URL to open.
5. Provides operational cells: log tailing, full diagnostics, and a safe stop
   switch — matching the old notebook's operational conventions.

> Run this top-to-bottom on a fresh RunPod RTX 5090 pod (Ubuntu, CUDA 12.8
> base image). Edit the **Parameters** cell first. You must have a Hugging
> Face account **approved for `nvidia/personaplex-7b-v1`** with a read token.


In [ ]:
!git clone https://github.com/MoshiHead/new_ref_data_create_and_lora_training.git

## Step -1 — Get the updated code onto this pod

This package (`speech2avatar-new-code`) is distributed as a folder/zip, not a
public git URL. Before running the Parameters cell, get the code onto the pod
using **one** of these methods, then point `PROJECT_ROOT` at it:

- **Jupyter upload**: use the file browser to upload the `speech2avatar-new-code`
  folder (or a zip of it) into `/workspace/`, e.g. as
  `/workspace/speech2avatar_imtalker_personaplex_8998_try_vad2.zip`.
- **`runpodctl send` / `scp`** from your local machine into `/workspace/`.
- If you have since pushed this code to a **private git repo**, set
  `GIT_REPO_URL` (and `GIT_BRANCH`) in the Parameters cell below instead —
  Step 0 will clone it automatically.

If you uploaded a **zip file**, set `UPLOAD_ZIP_PATH` in the Parameters cell
to its path — Step 0 will unzip it into `PROJECT_ROOT` automatically.


In [ ]:
import os

# --- Project location -------------------------------------------------------
# If PROJECT_ROOT already contains a valid speech2avatar-new-code checkout, it
# is used as-is. Otherwise Step 0 tries, in order: unzip UPLOAD_ZIP_PATH (if
# set and it exists), clone GIT_REPO_URL (if set), else fail with instructions.
PROJECT_ROOT = "/workspace/speech2avatar"
UPLOAD_ZIP_PATH = ""  # set "" to disable
GIT_REPO_URL = "https://github.com/MoshiHead/new_ref_data_create_and_lora_training.git"          # e.g. "https://github.com/<you>/speech2avatar-new-code.git" (leave blank if not using git)
GIT_BRANCH = "main"

# --- Toolchain ---------------------------------------------------------------
# prepare_imtalker_personaplex.sh defaults VENV_DIR to "$ROOT/.venv" when this
# env var is unset; keep that default unless you have a reason to change it.
VENV_DIR = f"{PROJECT_ROOT}/.venv"

# --- Service ------------------------------------------------------------------
HOST = "0.0.0.0"
PORT = 8998
CUDA_VISIBLE_DEVICES = "0"   # which physical GPU to bind, passed through to run_imtalker_personaplex.sh

# --- Runtime overrides consumed by run_imtalker_personaplex.sh (env vars) ---
VOICE_PROMPT = ""        # e.g. "Robert_5.pt" (script default) -- leave "" to use the script's own default
TEXT_PROMPT = ""         # e.g. "You are Robert from RB Labs. Answer every part clearly."
TEXT_PROMPT_FILE = ""    # e.g. f"{PROJECT_ROOT}/IMTalker/prompts/Robert_8998_default.txt" (script default)
PROMPT_CACHE = ""        # "" (use script default 0), or "1"/"0"

# --- Online search: STT -> router -> web search -> summary -> context injection ---
# Pipeline when ENABLE_SEARCH=True: STT transcribes what the user said, a small
# router decides whether the question needs live information, and only then does
# a web search run. If it does not, the model answers from its own knowledge and
# NOTHING is injected -- that path stays as fast as the plain conversational
# server. If it does, a thinking sound plays while the search runs, the results
# are summarized into one clean spoken sentence (numbers written out in words:
# "Three hundred twenty-five dollars", never "$325"), and that sentence is
# injected as a <ref> block into the LIVE context. The conversation is never
# reset. Consuming those <ref> tags is what the reference LoRA was trained for,
# which is why ENABLE_SEARCH also requires it to be downloaded (Step 3).
ENABLE_SEARCH = True
WEB_SEARCH_ENABLED = True          # let the router actually reach the web (needs the key below)
WEB_SEARCH_API_KEY = "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"            # Tavily/Serper/Bing key. Leave "" to be prompted securely.
WEB_SEARCH_PROVIDER = "tavily"     # tavily | serper | bing
ROUTER_THRESHOLD = "0.40"          # P(needs live data) at/above which a search fires. <0.5 on
                                   # purpose: an unnecessary search costs a couple of seconds and
                                   # is recoverable; a missed one is a wrong answer spoken aloud.
ROUTER_RULES = "1"                 # 1 = instant regex pre-pass first, so obvious cases cost 0ms
COMPRESSOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # ONE model both routes AND summarizes
SEARCH_MAX_FILLER_SEC = "6.0"      # give up waiting on search after this and answer anyway

# --- Latency controls -------------------------------------------------------
# MAX_INPUT_BUFFER_SEC is the important one. The GPU producer is pinned to real
# time by frame_q backpressure, so it can never drain a microphone backlog:
# without a cap, any transient stall becomes a PERMANENT session-long reply
# delay, which is what turns a ~2-4s pipeline into a 10-12s one. "" uses the
# launch script's default (2.0s). Set "0" only to reproduce the old unbounded
# behaviour.
MAX_INPUT_BUFFER_SEC = ""   # "" -> 6.0s (three generation chunks)
SPOKEN_FORM_NUMBERS = "0"    # "0" -> inject "$309.32" (digits). PersonaPlex reads
                            # digits aloud correctly by itself; spelling them out
                            # forces it to re-encode the value and it drops
                            # magnitudes ($309.32 was spoken back as $39.32).
                            # "1" spells them out anyway.
WAIT_FOR_USER = "1"         # "1" -> the model stays quiet until you actually speak.
                            # "0" lets it free-run from the system prompt at
                            # session start and read fragments of it aloud.
MAX_INPUT_BUFFER_SEC = ""   # "" -> 2.0. The ceiling on how STALE the audio the
                            # model answers may be, not just a drop threshold.
RENDER_SUB_BATCH = ""       # "" -> 8. VRAM first, latency second: the renderer's
                            # attention at resolution 64 needs
                            # batch x 8 heads x 4096 x 4096 x 4 bytes. Raise only if
                            # the vram_headroom line in system_*.log says you can.
JPEG_QUALITY = ""           # "" -> 82
PREBUFFER_CHUNKS = ""       # "" -> 1. Do NOT set 0: prebuffer_ready gates the media
                            # epoch the audio sender paces against, so releasing it
                            # before the first chunk exists anchors that clock ahead
                            # of the audio and causes an initial burst. It is a
                            # one-time session-start cost, not a per-turn one.
# Opt-in. Publishes each render sub-batch as soon as it is encoded instead of
# holding the whole 2s chunk, which takes render time off the reply path -- the
# biggest remaining win after MAX_INPUT_BUFFER_SEC. The cost is the atomic A/V
# publication guarantee, so enable it only after checking the per-chunk
# "render+jpeg=" numbers in the log are stable on your GPU.
INCREMENTAL_PUBLISH = ""    # "" -> 0 (off), "1" to enable

# --- Keeping the pipeline at real time ---------------------------------------
# These two decide whether the avatar can speak at all, not just how fast.
# The audio sender paces on a fixed 80ms grid, so if the model pipeline falls
# below 1.0x real time the browser's audio worklet under-runs continuously and
# outputs SILENCE -- while the text stream, and therefore the conversation log,
# still looks perfect. Watch the rtf= value on the [liveTryStudio] log lines.
#
# The reference LoRA runs UNMERGED, exactly as the old pipeline does. That is a
# proven configuration: the old pipeline ships the same adapter with merge
# disabled and holds real time on the same GPU. Merging is also not possible
# against a bnb-4bit base (peft adds a dense delta to a packed quantized blob
# and raises a shape mismatch), so "1" only takes the loud fallback path.
MERGE_REF_LORA = "0"        # "0" unmerged (default, matches old pipeline)
# STT_INLINE runs the 1B STT/VAD model inside the 80ms real-time step instead
# of on its own thread. Diagnostic only.
STT_INLINE = "0"            # "0" threaded (default), "1" inline
# Move the router/compressor off the GPU if VRAM or SM contention is the issue.
COMPRESSOR_DEVICE = "cuda"  # "cuda" or "cpu"

# --- Logging ------------------------------------------------------------------
# Two files per run, both with millisecond timestamps, written under LOG_DIR:
#   system_<session>.log       every model, where it was loaded from, LoRA paths,
#                              resolved config, video/render params, GPU, timing
#   detailed_<session>.log     per-turn report: what was asked, whether a search
#                              happened, what was searched, the results, the
#                              summary, exactly what was injected, the reply, and
#                              the duration of every stage
#   conversation_<session>.log the same events as a compact per-stage trace
LOG_DIR = ""   # "" -> f"{PROJECT_ROOT}/logs"

EXPECTED_TORCH_VERSION = "2.8.0+cu128"

# --- Startup wait behaviour ---------------------------------------------------
STARTUP_TIMEOUT_SEC = 900   # PersonaPlex (7B, 4-bit) + IMTalker model load can take a while
POLL_INTERVAL_SEC = 5
PREPARE_TIMEOUT_SEC = 7200  # multi-GB checkpoint downloads can take a long time on first run

# --- Derived paths ------------------------------------------------------------
IMTALKER_DIR = f"{PROJECT_ROOT}/IMTalker"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
PERSONAPLEX_BNB4_DIR = f"{CHECKPOINT_DIR}/personaplex_bnb4"
VENV_PYTHON = f"{VENV_DIR}/bin/python"
VENV_ACTIVATE = f"source {VENV_DIR}/bin/activate"

PREPARE_SCRIPT = f"{PROJECT_ROOT}/prepare_imtalker_personaplex.sh"
RUN_SCRIPT = f"{PROJECT_ROOT}/run_imtalker_personaplex.sh"

LOG_PATH = f"{PROJECT_ROOT}/live_server.log"
PID_PATH = f"{PROJECT_ROOT}/.run_imtalker_personaplex.pid"

# Structured-log locations (the launch script's own defaults, mirrored here so
# the operational cells below can find the files without re-deriving them).
STRUCTURED_LOG_DIR = LOG_DIR or f"{PROJECT_ROOT}/logs"
REF_LORA_DIR = f"{CHECKPOINT_DIR}/rag_lora"
STT_PKG_DIR = f"{CHECKPOINT_DIR}/stt"
THINKING_SOUND_PATH = f"{PROJECT_ROOT}/assets/ai-thinking-sound.wav"

os.makedirs("/workspace", exist_ok=True)
print("Parameters loaded.")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"VENV_DIR     = {VENV_DIR}")
print(f"HOST:PORT    = {HOST}:{PORT}")
print(f"ENABLE_SEARCH = {ENABLE_SEARCH}  (web_search={WEB_SEARCH_ENABLED}, "
      f"provider={WEB_SEARCH_PROVIDER}, router_threshold={ROUTER_THRESHOLD}, rules={ROUTER_RULES})")
print(f"STRUCTURED_LOG_DIR = {STRUCTURED_LOG_DIR}")


## Utilities

Shared helpers: a streaming shell runner with retries, a torch/CUDA probe
that always queries the **venv** interpreter, and port/log helpers. These
mirror the old notebook's utilities so the operational cells behave the same
way.


In [ ]:
import json as _json
import socket
import subprocess
import time


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False, timeout=None):
    last_returncode = None
    for attempt in range(1, retries + 1):
        if not quiet:
            print(f"$ {cmd}" + (f"   [attempt {attempt}/{retries}]" if retries > 1 else ""))
        proc = subprocess.Popen(
            cmd, shell=True, executable="/bin/bash", cwd=cwd,
            env=env or os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        start = time.time()
        for line in proc.stdout:
            print(line, end="")
            if timeout and (time.time() - start) > timeout:
                proc.kill()
                raise TimeoutError(f"Command exceeded timeout of {timeout}s: {cmd}")
        proc.wait()
        last_returncode = proc.returncode
        if last_returncode == 0:
            return 0
        print(f"[warn] command failed with exit code {last_returncode}")
        if attempt < retries:
            print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)
    if check:
        raise RuntimeError(f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}")
    return last_returncode


def get_torch_info():
    probe = (
        "import torch, json;"
        "print(json.dumps({"
        "'version': torch.__version__,"
        "'cuda_available': torch.cuda.is_available(),"
        "'cuda_version': torch.version.cuda,"
        "'device_name': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
        "}))"
    )
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError("Failed to query torch/CUDA state inside the venv")
    return _json.loads(out.stdout.strip().splitlines()[-1])


def port_listening(host, port, timeout=1.0):
    probe_host = "127.0.0.1" if host == "0.0.0.0" else host
    try:
        with socket.create_connection((probe_host, port), timeout=timeout):
            return True
    except OSError:
        return False


def find_pids_on_port(port):
    out = subprocess.run(
        f"fuser {port}/tcp 2>/dev/null || lsof -t -i:{port} 2>/dev/null",
        shell=True, executable="/bin/bash", capture_output=True, text=True,
    )
    return [p for p in out.stdout.split() if p.strip().isdigit()]


def tail_log(n=200):
    if not os.path.exists(LOG_PATH):
        print(f"[info] no log file yet at {LOG_PATH}")
        return
    with open(LOG_PATH, "r", errors="ignore") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


print("Utilities loaded.")


## Step 0 — Validate / obtain the project structure

Expects the `speech2avatar-new-code` layout:

```
speech2avatar/
  IMTalker/
  bundled_assets/Robert_5.pt
  prepare_imtalker_personaplex.sh
  run_imtalker_personaplex.sh
  live_8998.md
```

If `PROJECT_ROOT` is missing or incomplete, this cell tries, in order:
unzipping `UPLOAD_ZIP_PATH` into `PROJECT_ROOT`, then cloning `GIT_REPO_URL`.
If neither is usable it fails fast with a clear diagnostic (see the "Step -1"
markdown cell above for how to get the code onto the pod).


In [ ]:
REQUIRED_STRUCTURE = [
    "IMTalker",
    "IMTalker/requirement.txt",
    "bundled_assets/Robert_5.pt",
    "prepare_imtalker_personaplex.sh",
    "run_imtalker_personaplex.sh",
]

# Checked by name rather than assumed: a missing IMTalker/search_helpers.py is
# the single most likely cause of search silently not loading, and the server
# degrades gracefully instead of failing loudly when it is absent -- so the
# check has to happen here, not at runtime.
SEARCH_STRUCTURE = [
    "IMTalker/search_helpers.py",       # routing, web search, compression, STT loader
    "IMTalker/conversation_logger.py",  # the conversation log
    "IMTalker/system_logger.py",        # the system-information log
    "IMTalker/speech_text.py",          # spoken-form normalization of injected summaries
    "assets/ai-thinking-sound.wav",     # played only while a search is actually running
]


def structure_ok(root):
    return all(os.path.exists(os.path.join(root, rel)) for rel in REQUIRED_STRUCTURE)


if not structure_ok(PROJECT_ROOT):
    if UPLOAD_ZIP_PATH and os.path.exists(UPLOAD_ZIP_PATH):
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; unzipping {UPLOAD_ZIP_PATH}")
        os.makedirs(PROJECT_ROOT, exist_ok=True)
        run(f"unzip -o -q {UPLOAD_ZIP_PATH} -d {PROJECT_ROOT}")
        # A zip that contains a single top-level folder unpacks one level too deep; flatten it.
        entries = [e for e in os.listdir(PROJECT_ROOT) if not e.startswith(".")]
        if not structure_ok(PROJECT_ROOT) and len(entries) == 1:
            inner = os.path.join(PROJECT_ROOT, entries[0])
            if os.path.isdir(inner) and structure_ok(inner):
                run(f"shopt -s dotglob && mv {inner}/* {PROJECT_ROOT}/ && rmdir {inner}")
    elif GIT_REPO_URL:
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; cloning {GIT_REPO_URL} (branch {GIT_BRANCH})")
        os.makedirs(os.path.dirname(PROJECT_ROOT.rstrip("/")) or "/", exist_ok=True)
        if os.path.exists(PROJECT_ROOT) and not os.listdir(PROJECT_ROOT):
            os.rmdir(PROJECT_ROOT)
        run(f"git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {PROJECT_ROOT}", retries=3, retry_delay=10)
    else:
        raise RuntimeError(
            f"PROJECT_ROOT '{PROJECT_ROOT}' does not contain a valid speech2avatar-new-code checkout "
            f"(missing one of {REQUIRED_STRUCTURE}), UPLOAD_ZIP_PATH does not exist, and GIT_REPO_URL is "
            f"empty. Upload the code (see the 'Step -1' cell above) or set one of those parameters."
        )

print(f"Validating structure under {PROJECT_ROOT}:")
status_ok = True
for rel in REQUIRED_STRUCTURE:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    status_ok = status_ok and exists
    print(f"  [{'OK' if exists else 'MISSING'}] {rel}")

if not status_ok:
    raise RuntimeError(f"Required speech2avatar-new-code structure is incomplete under {PROJECT_ROOT}")

if ENABLE_SEARCH:
    print("\nOnline-search modules:")
    missing_search = []
    for rel in SEARCH_STRUCTURE:
        exists = os.path.exists(os.path.join(PROJECT_ROOT, rel))
        if not exists:
            missing_search.append(rel)
        print(f"  [{'OK' if exists else 'MISSING'}] {rel}")
    if missing_search:
        raise RuntimeError(
            f"ENABLE_SEARCH is True but this checkout is missing {missing_search}. "
            f"Update the code on the pod, or set ENABLE_SEARCH = False in the Parameters cell."
        )

run(f"chmod +x {PROJECT_ROOT}/prepare_imtalker_personaplex.sh {PROJECT_ROOT}/run_imtalker_personaplex.sh")
print("[ok] project structure validated")


## Step 1 — GPU / driver check (pre-install)

Confirms a GPU and NVIDIA driver are visible to the container before spending
time on installation and multi-GB downloads. The RTX 5090 name check is a
warning, not a hard failure, in case the pod surfaces a slightly different
GPU string.


In [ ]:
smi = subprocess.run("nvidia-smi", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(smi.stdout)
if smi.returncode != 0:
    print(smi.stderr)
    raise RuntimeError(
        "nvidia-smi failed — no NVIDIA driver/GPU visible to this container. "
        "Check the RunPod GPU pod template and that you selected an RTX 5090 instance."
    )

if "5090" in smi.stdout:
    print("[ok] RTX 5090 detected by nvidia-smi")
else:
    print("[warn] '5090' not found in nvidia-smi output — continuing, but confirm the GPU type for this pod")


## Step 2 — Hugging Face token

Token is requested securely via `getpass` and passed **only** as the
`--hf-token` argument to `prepare_imtalker_personaplex.sh` for this process —
it is never written to a cell, a file, or hardcoded. You need a Hugging Face
account **approved for `nvidia/personaplex-7b-v1`** with a read token (see
`live_8998.md`'s Requirements section).


In [ ]:
import getpass

HF_TOKEN = ""
for attempt in range(1, 4):
    token = '_tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'
    if token:
        HF_TOKEN = 'hf' + token
        break
    print("[warn] empty token entered, try again")

if not HF_TOKEN:
    raise RuntimeError("No Hugging Face token entered after 3 attempts.")

print("[ok] token captured for this session (not displayed or persisted)")


### Step 2b — Web search API key (only when `ENABLE_SEARCH` and `WEB_SEARCH_ENABLED` are both True)

Requested securely via `getpass` and held in this process only. **Never hardcode it in a cell**: notebooks get committed, shared, and pasted into issues, and a key pasted here leaks with them.


In [ ]:
import getpass as _getpass

if ENABLE_SEARCH and WEB_SEARCH_ENABLED:
    key = WEB_SEARCH_API_KEY or _getpass.getpass(
        f"Web search API key for provider '{WEB_SEARCH_PROVIDER}' (input hidden): "
    ).strip()
    if not key:
        raise RuntimeError(
            "WEB_SEARCH_ENABLED is True but no web search API key was provided. "
            "Set WEB_SEARCH_ENABLED = False to let the router fall back to the "
            "model's own knowledge for questions that need live data."
        )
    WEB_SEARCH_API_KEY = key
    print("[ok] web search key set for this process (never printed, never written to a cell)")
elif ENABLE_SEARCH:
    print("[info] ENABLE_SEARCH is True but WEB_SEARCH_ENABLED is False --")
    print("       the router still decides; questions needing live data fall back")
    print("       to the model's own knowledge instead of hanging.")
else:
    print("Search disabled - skipping web-search key prompt.")


## Step 3 — Run `prepare_imtalker_personaplex.sh`

This single script performs everything the old notebook did across many
cells: apt packages (`python3.11`, `python3.11-venv`, `ffmpeg`, `git`,
`git-lfs`, `htop`, `tmux`, `curl`, `ca-certificates`, `build-essential`),
`git lfs install`, the `.venv` creation, the **pinned** `torch==2.8.0+cu128`
stack, `IMTalker/requirement.txt`, the pinned extras
(`huggingface_hub[cli]==0.36.2`, `hf_transfer`, `tensorboard`,
`sphn==0.2.1`, `einops`, `sentencepiece`, `aiohttp==3.14.3`, `av==17.1.0`,
`aiortc==1.15.0`, `bitsandbytes==0.50.0`), Hugging Face auth, every checkpoint
download (IMTalker renderer/wav2vec2, 2s generator + adapter, blink motion,
PersonaPlex bnb4 weights + Mimi/tokenizer + voices + the bundled
`Robert_5.pt` voice), the `--no-deps` Moshi install, and finally the script's
own preflight (`run_imtalker_personaplex.sh --check-only`, including SHA-256
verification of the protected runtime files).

The notebook never reimplements this script's steps — it only invokes it
and streams its output. This can take a long time on a fresh pod (multi-GB
downloads); `PREPARE_TIMEOUT_SEC` bounds it. Hugging Face downloads resume
automatically on re-run if interrupted.


In [ ]:
prepare_env = os.environ.copy()
prepare_env["SPEECH2AVATAR_ROOT"] = PROJECT_ROOT
prepare_env["VENV_DIR"] = VENV_DIR

# --with-search additionally installs the peft/transformers pins, an ISOLATED
# copy of the upstream Kyutai moshi package for the STT submodel (it cannot go
# into site-packages: PersonaPlex ships a fork under the same import name), and
# the reference LoRA adapter that teaches PersonaPlex to consume injected
# <lookup>/<ref> context. Omitting it leaves the server usable with
# ENABLE_SEARCH=0 only.
search_flag = " --with-search" if ENABLE_SEARCH else ""

run(
    f'bash "{PREPARE_SCRIPT}" --hf-token "{HF_TOKEN}"{search_flag}',
    cwd=PROJECT_ROOT, env=prepare_env, retries=1, timeout=PREPARE_TIMEOUT_SEC,
)
print("[ok] prepare_imtalker_personaplex.sh completed (includes the built-in --check-only preflight)")

if ENABLE_SEARCH:
    print("\nSearch assets:")
    for label, path in (
        ("reference LoRA weights", f"{REF_LORA_DIR}/lora/adapter_model.safetensors"),
        ("reference LoRA config", f"{REF_LORA_DIR}/lora/adapter_config.json"),
        ("isolated STT package", f"{STT_PKG_DIR}/moshi/__init__.py"),
        ("thinking sound", THINKING_SOUND_PATH),
    ):
        exists = os.path.exists(path)
        size = f"{os.path.getsize(path) / (1 << 20):.1f} MB" if exists else "-"
        print(f"  [{'OK' if exists else 'MISSING'}] {label:<24} {path}  {size}")


## Step 4 — Post-install GPU / CUDA confirmation

Independent double-check (beyond the script's own preflight) that the venv's
torch matches the required pin and CUDA is available, using the same probe
the old notebook used.


In [ ]:
torch_info = get_torch_info()
print(f"[info] torch={torch_info['version']} cuda_available={torch_info['cuda_available']} "
      f"cuda_version={torch_info['cuda_version']} device={torch_info['device_name']}")

if torch_info["version"] != EXPECTED_TORCH_VERSION:
    raise RuntimeError(f"torch pin violated: {torch_info['version']} != {EXPECTED_TORCH_VERSION}")
if not torch_info["cuda_available"]:
    raise RuntimeError("torch.cuda.is_available() is False inside the venv after installation.")
if torch_info["device_name"] and "5090" in torch_info["device_name"]:
    print(f"[ok] GPU confirmed: {torch_info['device_name']}")
else:
    print(f"[warn] GPU device name '{torch_info['device_name']}' does not mention 5090 — continuing anyway")


## Step 5 — Pre-launch port check

`run_imtalker_personaplex.sh` refuses to start (rather than recovering) if
the port is already occupied. This cell mirrors the old notebook's recovery
behaviour: if the port is held by a *previous run of this same notebook*
(tracked via `PID_PATH`), it is terminated automatically; if held by anything
else, the cell fails fast with instructions rather than guessing.


In [ ]:
def check_port_and_recover(port):
    if not port_listening("127.0.0.1", port):
        print(f"[ok] port {port} is free")
        return
    pids_on_port = find_pids_on_port(port)
    prior_pid = None
    if os.path.exists(PID_PATH):
        prior_pid = open(PID_PATH).read().strip()
    if prior_pid and prior_pid in pids_on_port:
        print(f"[recovery] port {port} held by a previous notebook-launched server (pid {prior_pid}); terminating it")
        subprocess.run(["kill", "-9", prior_pid])
        time.sleep(2)
        if port_listening("127.0.0.1", port):
            raise RuntimeError(f"Port {port} still in use after terminating prior pid {prior_pid}")
        print(f"[ok] port {port} freed")
    else:
        raise RuntimeError(
            f"Port {port} is already in use by pid(s) {pids_on_port}, which were not started by this "
            f"notebook. Stop that process manually (or choose a different PORT) before launching."
        )


check_port_and_recover(PORT)


## Step 6 — Inspect `run_imtalker_personaplex.sh` and resolve runtime overrides

The notebook never reimplements `run_imtalker_personaplex.sh`'s argument
list — it reads and prints the actual script, then prepares only the
environment-variable overrides the script already supports (`PORT`,
`CUDA_VISIBLE_DEVICES`, `VOICE_PROMPT`, `TEXT_PROMPT` / `TEXT_PROMPT_FILE`,
`PROMPT_CACHE`, plus `SPEECH2AVATAR_ROOT` / `VENV_DIR`).


In [ ]:
with open(RUN_SCRIPT) as f:
    run_script_contents = f.read()
print(run_script_contents)


In [ ]:
env_overrides = {
    "SPEECH2AVATAR_ROOT": PROJECT_ROOT,
    "VENV_DIR": VENV_DIR,
    "PORT": str(PORT),
    "CUDA_VISIBLE_DEVICES": str(CUDA_VISIBLE_DEVICES),
    "LOG_DIR": STRUCTURED_LOG_DIR,
}
if VOICE_PROMPT:
    env_overrides["VOICE_PROMPT"] = VOICE_PROMPT
if TEXT_PROMPT:
    env_overrides["TEXT_PROMPT"] = TEXT_PROMPT
if TEXT_PROMPT_FILE:
    env_overrides["TEXT_PROMPT_FILE"] = TEXT_PROMPT_FILE
if PROMPT_CACHE:
    env_overrides["PROMPT_CACHE"] = PROMPT_CACHE

# Latency controls: every one of these is already defaulted inside the launch
# script, so an empty value here means "use the script's default" rather than
# "unset".
for var, value in (
    ("MAX_INPUT_BUFFER_SEC", MAX_INPUT_BUFFER_SEC),
    ("RENDER_SUB_BATCH", RENDER_SUB_BATCH),
    ("SPOKEN_FORM_NUMBERS", SPOKEN_FORM_NUMBERS),
    ("WAIT_FOR_USER", WAIT_FOR_USER),
    ("MAX_INPUT_BUFFER_SEC", MAX_INPUT_BUFFER_SEC),
    ("JPEG_QUALITY", JPEG_QUALITY),
    ("PREBUFFER_CHUNKS", PREBUFFER_CHUNKS),
    ("INCREMENTAL_PUBLISH", INCREMENTAL_PUBLISH),
):
    if value:
        env_overrides[var] = str(value)

if ENABLE_SEARCH:
    env_overrides["ENABLE_SEARCH"] = "1"
    env_overrides["ROUTER_THRESHOLD"] = ROUTER_THRESHOLD
    env_overrides["ROUTER_RULES"] = ROUTER_RULES
    env_overrides["COMPRESSOR_MODEL"] = COMPRESSOR_MODEL
    env_overrides["COMPRESSOR_DEVICE"] = COMPRESSOR_DEVICE
    env_overrides["MERGE_REF_LORA"] = MERGE_REF_LORA
    env_overrides["STT_INLINE"] = STT_INLINE
    env_overrides["SEARCH_MAX_FILLER_SEC"] = SEARCH_MAX_FILLER_SEC
    env_overrides["WEB_SEARCH_PROVIDER"] = WEB_SEARCH_PROVIDER
    if WEB_SEARCH_ENABLED and WEB_SEARCH_API_KEY:
        env_overrides["WEB_SEARCH_ENABLED"] = "1"
        env_overrides["WEB_SEARCH_API_KEY"] = WEB_SEARCH_API_KEY
    else:
        env_overrides["WEB_SEARCH_ENABLED"] = "0"
else:
    env_overrides["ENABLE_SEARCH"] = "0"

print("Resolved environment overrides for run_imtalker_personaplex.sh:")
for k, v in env_overrides.items():
    # The key is the one value in here that must never be echoed: this output
    # is saved into the .ipynb and travels with it.
    shown = "<set>" if "API_KEY" in k else v
    print(f"  {k}={shown}")


## Step 7 — Launch the live server

`run_imtalker_personaplex.sh` runs its preflight again and then `exec`s the
server in the foreground, so this notebook launches it as a background
process (like the old notebook did with `run_live.sh`), logs to
`live_server.log`, and records the PID for the stop/recovery cells below.
Because the script uses `exec`, the recorded PID stays valid for the actual
Python server process (Unix `exec` replaces the process image but keeps the
PID).


In [ ]:
launch_env = os.environ.copy()
launch_env.update(env_overrides)

log_file = open(LOG_PATH, "w")
live_proc = subprocess.Popen(
    ["bash", RUN_SCRIPT], cwd=PROJECT_ROOT, env=launch_env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
with open(PID_PATH, "w") as f:
    f.write(str(live_proc.pid))

print(f"[ok] launched live server: pid={live_proc.pid}")
print(f"     log file: {LOG_PATH}")
print(f"     pid file: {PID_PATH}")


## Step 8 — Wait for healthy startup

Polls the log and the listening port simultaneously. Healthy markers are
drawn from the script's own preflight banner (`Preflight OK: try_vad2, ...`)
and the standard Uvicorn startup banner. On timeout or early process exit, it
dumps the log tail and common-failure hints instead of hanging forever.


In [ ]:
HEALTHY_MARKERS = {
    "preflight OK": "Preflight OK: try_vad2",
    "Uvicorn running on host:port": f"Uvicorn running on http://{HOST}:{PORT}",
}

# Reported after startup rather than waited on. Search is additive by design:
# every one of these components degrades to "answer from the model's own
# knowledge" instead of failing the server, so a missing one must not block a
# healthy launch -- but it must not pass unnoticed either.
SEARCH_MARKERS = {
    "reference LoRA loaded": "reference LoRA loaded from",
    "STT model loaded": "STT model loaded:",
    "router/compressor ready": "[search_helpers][compressor] ready",
    "query router ready": "query router ready",
    "reference LoRA merged": "merged=True",
    "STT on its own thread": "worker thread started",
    "thinking sound loaded": "thinking sound loaded:",
}

ERROR_HINTS = {
    "CUDA out of memory": "GPU out of memory — reduce concurrent load or confirm the pod truly has an RTX 5090 with enough VRAM",
    "Traceback (most recent call last)": "a Python exception occurred during startup — see the traceback above in the log",
    "Missing required file": "a referenced checkpoint/asset path is missing — re-run Step 3 (prepare script)",
        "Checksum check: MODIFIED": "a pinned model/runtime file does not match its expected hash — usually a partial download; re-run Step 3. If you replaced it on purpose, update its hash in run_imtalker_personaplex.sh",
        "Checksum check: MISSING": "a pinned model/runtime file is absent — re-run Step 3 to download it",
    "CUDA error": "a CUDA/driver mismatch occurred — re-check Step 1 (nvidia-smi) and the torch CUDA build",
    "Port": "the port was taken by another process after the Step 5 check — re-run Step 5",
}


def wait_for_healthy(timeout=STARTUP_TIMEOUT_SEC, poll_interval=POLL_INTERVAL_SEC):
    start = time.time()
    last_print = 0.0
    while time.time() - start < timeout:
        if live_proc.poll() is not None:
            tail_log(150)
            raise RuntimeError(
                f"live server process exited early with code {live_proc.returncode}; see log tail above"
            )

        log_text = ""
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", errors="ignore") as f:
                log_text = f.read()

        markers_ok = {label: (needle in log_text) for label, needle in HEALTHY_MARKERS.items()}
        port_ok = port_listening("127.0.0.1", PORT)

        if all(markers_ok.values()) and port_ok:
            print("[ok] live server is healthy")
            for label, ok in markers_ok.items():
                print(f"  [OK] {label}")
            print(f"  [OK] port {PORT} listening")
            return True

        if time.time() - last_print > 15:
            elapsed = int(time.time() - start)
            print(f"[..] waiting ({elapsed}s/{timeout}s) markers={markers_ok} port_listening={port_ok}")
            last_print = time.time()

        time.sleep(poll_interval)

    print("[error] timed out waiting for healthy startup; log tail:")
    tail_log(150)
    log_text = ""
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            log_text = f.read()
    for needle, hint in ERROR_HINTS.items():
        if needle in log_text:
            print(f"[diagnosis] found '{needle}' in log -> {hint}")
    raise TimeoutError("Live server did not become healthy within the timeout; see diagnostics above")


wait_for_healthy()

if ENABLE_SEARCH:
    with open(LOG_PATH, "r", errors="ignore") as f:
        startup_log = f.read()
    print("\nOnline-search components:")
    for label, needle in SEARCH_MARKERS.items():
        print(f"  [{'UP' if needle in startup_log else 'DOWN'}] {label}")
    if "could not merge the reference LoRA" in startup_log:
        print("  [WARN] the reference LoRA could NOT be merged and is running unmerged.")
        print("         Every model step now pays extra matmuls on nearly every projection,")
        print("         which can push the pipeline below real time and silence the avatar.")
        print("         Watch the rtf= values; if they sit below 1.00, upgrade peft or")
        print("         relaunch with ENABLE_SEARCH = False.")
    if "search disabled" in startup_log:
        print("  [note] the server logged 'search disabled' -- grep live_server.log for the reason;")
        print("         the avatar still works, it just answers from its own knowledge only.")


## Step 9 — HTTP confirmation

Final confirmation that the FastAPI/Uvicorn app is actually serving HTTP on
`0.0.0.0:8998`, plus the two health-check URLs documented in `live_8998.md`
(`/` and `/assets/robert_idle_10s.mp4`), not just that the port is open.


In [ ]:
import urllib.error
import urllib.request

for path in ["/", "/assets/robert_idle_10s.mp4"]:
    url = f"http://127.0.0.1:{PORT}{path}"
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            print(f"[ok] HTTP {resp.status} from {url}")
    except urllib.error.URLError as e:
        tail_log(80)
        raise RuntimeError(f"HTTP check failed against {url}: {e}")

with open(LOG_PATH, "r", errors="ignore") as f:
    final_log = f.read()
assert f"Uvicorn running on http://{HOST}:{PORT}" in final_log, "Uvicorn startup banner not found in log"

print()
print("=" * 72)
print("SUCCESS: PersonaPlex + IMTalker (updated / try_vad2) live server is running")
print(f"  Internal: http://{HOST}:{PORT}")
print(f"  Open port {PORT} through your RunPod pod's proxy/port mapping to access")
print(f"  the browser UI (index_v3_binary_fullscreen_robot_try_vad2.html is served at /).")
print(f"  PID: {open(PID_PATH).read().strip()}   Log: {LOG_PATH}")
print("=" * 72)


## Operational cells (run any time)

These are safe to re-run independently after the server is up.


In [ ]:
# Tail the live server log
tail_log(200)


In [ ]:
# The two structured logs. Both carry millisecond timestamps, so a slow stage
# in the conversation log can be lined up against the system log by time alone.
def show_structured_logs(system_lines=80, turns=3):
    import glob

    if not os.path.isdir(STRUCTURED_LOG_DIR):
        print(f"[info] no structured logs yet at {STRUCTURED_LOG_DIR}")
        return

    def newest(pattern):
        matches = sorted(glob.glob(os.path.join(STRUCTURED_LOG_DIR, pattern)))
        return matches[-1] if matches else None

    sys_log = newest("system_*.log")
    detail_log = newest("detailed_*.log")

    print("=" * 78)
    print(f"SYSTEM LOG  {sys_log or '(none yet)'}")
    print("=" * 78)
    if sys_log:
        print("".join(open(sys_log, errors="ignore").readlines()[-system_lines:]))

    print("=" * 78)
    print(f"CONVERSATION LOG (last {turns} turn reports)  {detail_log or '(none yet)'}")
    print("=" * 78)
    if detail_log:
        text = open(detail_log, errors="ignore").read()
        blocks = text.split("COMPLETE TURN REPORT")
        if len(blocks) > 1:
            for block in blocks[-turns:]:
                print("COMPLETE TURN REPORT" + block)
        else:
            print(text[-4000:])
            print("\n[info] no completed turn reports yet -- a turn's report is written once the")
            print("       user speaks again, which is the first moment its reply is known to be done.")


show_structured_logs()


In [ ]:
# ---- If the avatar was silent, run this ----------------------------------
# Silence is almost always the GPU producer dying, and the ONLY place its
# traceback appears is the server's stdout. Both structured logs keep looking
# healthy when this happens, because Moshi, STT and search are all still alive.
import glob, os, re, subprocess

SERVER_LOG = "/workspace/speech2avatar/live_server.log"

MARKERS = [
    ("THREAD-DIED",      "the producer crashed -- the traceback below is the root cause"),
    ("THREAD-RESTART",   "it crashed and was restarted"),
    ("RENDER-OOM",       "renderer ran out of VRAM; lower RENDER_SUB_BATCH"),
    ("VRAM",             "headroom once every model was resident"),
    ("[GPU][chunk#",     "chunks the producer actually emitted"),
    ("NATIVE-AUDIO",     "audio packets actually sent to the browser"),
]

if not os.path.exists(SERVER_LOG):
    print(f"no {SERVER_LOG} -- start the server with ./run_imtalker_personaplex.sh 2>&1 | tee {SERVER_LOG}")
else:
    text = open(SERVER_LOG, errors="replace").read()
    for marker, why in MARKERS:
        hits = [l for l in text.splitlines() if marker in l]
        print(f"{marker:16s} {len(hits):5d}  {why}")
        for l in hits[:3]:
            print("      ", l[:160])
    print()
    if "THREAD-DIED" in text:
        i = text.index("THREAD-DIED")
        print("=" * 70)
        print("FIRST THREAD DEATH -- this is what silenced the avatar")
        print("=" * 70)
        print(text[i:i + 3000])
    else:
        print("No thread died. If it was still silent, check rtf= and")
        print("realtime_factor_low in the system log instead.")


In [ ]:
# Full diagnostics — safe to run any time, including during a failure
def run_full_diagnostics():
    print("== GPU ==")
    run("nvidia-smi", check=False)

    print("\n== Torch / CUDA (venv) ==")
    try:
        print(get_torch_info())
    except Exception as e:
        print(f"[error] torch check failed: {e}")

    print(f"\n== Port {PORT} ==")
    print(f"listening: {port_listening('127.0.0.1', PORT)}  pids: {find_pids_on_port(PORT)}")

    print("\n== Preflight (--check-only) ==")
    check_env = os.environ.copy()
    check_env["SPEECH2AVATAR_ROOT"] = PROJECT_ROOT
    check_env["VENV_DIR"] = VENV_DIR
    run(f'bash "{RUN_SCRIPT}" --check-only', cwd=PROJECT_ROOT, env=check_env, check=False)

    print("\n== pip check (venv) ==")
    run(f"{VENV_ACTIVATE} && pip check", check=False)

    print("\n== Log tail ==")
    tail_log(100)


run_full_diagnostics()


In [ ]:
# Stop switch — set STOP_SERVER = True and re-run this cell to terminate the live server.
STOP_SERVER = False


def stop_server():
    if not os.path.exists(PID_PATH):
        print("[info] no pid file found; nothing to stop")
        return
    pid = open(PID_PATH).read().strip()
    print(f"[stop] terminating live server pid {pid}")
    subprocess.run(["pkill", "-9", "-P", pid], check=False)
    subprocess.run(["kill", "-9", pid], check=False)
    os.remove(PID_PATH)
    print("[ok] server stopped")


if STOP_SERVER:
    stop_server()
else:
    print("STOP_SERVER is False — set it to True above and re-run this cell to stop the server.")
